# 建设评估问题集

评估问题可以分为三类：日常必须稳定的问题、过去答错的问题，以及专门检查边界的困难问题。名称可以不同，但用途要分清。

本页是问题集建设的方法说明和字段运行示例；把问题登记进来不等于它已经证明某个方法有效。

In [1]:
import sys
from pathlib import Path

course_root = next(
    (folder for folder in (Path.cwd(), *Path.cwd().parents)
     if (folder / "data" / "dataset" / "manifest.json").is_file()),
    None,
)
if course_root is None:
    raise FileNotFoundError("没有找到 canonical 数据包，请从 C7 根目录运行。")
if str(course_root) not in sys.path:
    sys.path.insert(0, str(course_root))
from common.dataset import load_dataset

package = load_dataset(course_root / "data" / "dataset")
cases = package["queries"]
all_query_ids = {item["query_id"] for item in cases}
teaching_ids = set(package["manifest"]["usage_policy"]["current_case_ids"])
finetune_ids = {item["query_id"] for item in package["finetune_pairs"]}
method_ids = {usage[role] for usage in package["method_cases"].values() for role in ("main", "check")}
remaining_teaching_ids = teaching_ids - method_ids

assert method_ids <= teaching_ids
assert teaching_ids.isdisjoint(finetune_ids)
assert all_query_ids == teaching_ids | finetune_ids
print("canonical 问题总数：", len(cases))
print("教学问题：", len(teaching_ids))
print("教学问题中已直接用于方法展示：", len(method_ids))
print("教学问题中留给后续检查：", len(remaining_teaching_ids))
print("Embedding 微调问法（独立 train/dev/test 划分）：", len(finetune_ids))
print("微调问法不计入教学方法的后续检查题。")
print("编号是否唯一：", len(all_query_ids) == len(cases))

canonical 问题总数： 233
教学问题： 70
教学问题中已直接用于方法展示： 66
教学问题中留给后续检查： 4
Embedding 微调问法（独立 train/dev/test 划分）： 163
微调问法不计入教学方法的后续检查题。
编号是否唯一： True


每个问题至少要保留用户原问、答案所需的原文、原始检索结果和应用场景。模型生成的问题只能作为候选，需要人工确认原文确实能回答，并删除重复和泄露答案的问法。

当线上出现新错误时，先修正资料和参考答案，再加入问题集。问题只是登记了，不等于某个方法已经在它上通过。



## 三类问题集、分类和自动生成

评估集工程至少要分清三类用途：日常稳定问题（每次改动都跑）、历史错误问题（修复后仍保留，防止回归）和边界困难问题（多跳、模糊、拒答、权限等）。它们不能混成一个平均分：第一类看稳定性，第二类看修复是否保持，第三类用来发现能力上限。

每条记录建议保留问题编号、原始问题、参考答案或必要证据、场景分组、期望页、资料版本、来源和人工审核状态。还可以按事实、推理、含糊、无法回答、领域、权限和多轮状态分组。模型自动生成的问题只能当候选，必须检查是否能由原文回答、是否泄露答案、是否与已有问题重复。

当前 canonical 数据包共有 233 个问题：70 个教学问题和 163 个 Embedding 微调问法。70 个教学问题中，66 个直接用于各方法的主例与复查，另 4 个留给后续教学检查；163 个微调问法使用独立的 train/dev/test 划分，不能把它们与这 4 个教学问题合称为“167 个后续检查题”。这些数量由上方代码从当前 manifest、method_cases 和 finetune_pairs 交叉计算，数据版本变化后应重新运行。


In [2]:
import re

def validate_eval_case(item):
    required = {"id", "query", "expected_pages"}; missing = sorted(required - set(item))
    query = str(item.get("query", "")).strip()
    interaction = item.get("interaction") or {}
    is_unanswerable = (interaction.get("mode") in {"guardrail", "access_filter"}
                       or item.get("failure_mode") == "hallucination_risk")
    has_expected_evidence = bool(set(item.get("expected_pages", [])))
    return {"ok": not missing and bool(query) and (has_expected_evidence or is_unanswerable),
            "missing": missing,
            "is_unanswerable": is_unanswerable,
            "has_answer_leak": any(token in query for token in ("答案是", "正确页为", "参考答案"))}

def infer_bucket(item):
    """没有显式 bucket 时，从案例的交互方式和失败类型给出可复查的初始桶。"""
    if item.get("bucket"):
        return item["bucket"]
    interaction = item.get("interaction") or {}
    if interaction.get("mode") == "guardrail" or item.get("failure_mode") == "hallucination_risk":
        return "unanswerable"
    if item.get("failure_mode") in {"multihop", "context_incomplete"}:
        return "reasoning"
    if interaction.get("mode") != "single_turn":
        return "ambiguous"
    return "factual"

def bucket_cases(cases):
    buckets = {"factual": [], "reasoning": [], "ambiguous": [], "unanswerable": []}
    for item in cases: buckets.setdefault(infer_bucket(item), []).append(item)
    return buckets

QUESTION_GENERATION_PROMPT = """根据给定原文生成一个不能直接抄标题、但能由原文回答的问题。
同时返回需要核对的原文片段、问题类型和难度。不要输出原文没有的信息，不要在问题中泄露答案。"""

def accept_generated_candidate(candidate, existing_queries, source_text):
    query = str(candidate.get("query", "")).strip()
    if not query or query in existing_queries: return False
    if any(marker in query for marker in ("答案是", "第几页", "参考答案")): return False
    terms = [t for t in re.findall(r"[一-鿿A-Za-z0-9]+", query) if len(t) > 1]
    return any(term in source_text for term in terms)

# 生成候选需要外部 LLM；入集前仍应人工检查 evidence、答案范围和重复问题。
sample_bucketed = bucket_cases([
    {"id": "f1", "query": "什么是信息增益？", "expected_pages": [12], "interaction": {"mode": "single_turn"}},
    {"id": "u1", "query": "资料中有没有 CUDA 版本建议？", "expected_pages": [],
     "interaction": {"mode": "guardrail"}, "failure_mode": "hallucination_risk"},
])
print(
    "问题分类统计：" + "，".join(
        f"{bucket}类 {len(items)} 条" for bucket, items in (("事实", sample_bucketed["factual"]),
                                                     ("推理", sample_bucketed["reasoning"]),
                                                     ("含糊", sample_bucketed["ambiguous"]),
                                                     ("资料不足", sample_bucketed["unanswerable"]))
)
)

问题分类统计：事实类 1 条，推理类 0 条，含糊类 0 条，资料不足类 1 条


## 生成候选、审核入集和持续更新

自动生成只负责提出候选，不负责把候选变成事实。入集前逐条检查：原文是否真的能回答、是否泄露答案、证据范围是否清楚、是否与现有问题重复、问题属于哪一类，以及资料或权限变化后答案是否仍有效。新功能、线上错误和资料更新都是补题时机；历史错误修好后仍保留，不能删掉来让平均分变好。

当前教程问题集中的问题编号在后续评估中不会变化。下面的函数只生成带版本和审核状态的候选记录，不写回现有映射，也没有调用外部模型。

In [3]:
from datetime import date

def make_candidate_record(candidate, source_ref, set_version, generated_by="human"):
    """将候选统一成待审核记录；审核前不应加入回归统计。"""
    return {
        "id": str(candidate.get("id", "candidate-unknown")),
        "query": str(candidate.get("query", "")).strip(),
        "bucket": infer_bucket(candidate),
        "expected_pages": list(candidate.get("expected_pages", [])),
        "evidence": candidate.get("evidence", []),
        "source_ref": source_ref,
        "set_version": set_version,
        "generated_by": generated_by,
        "review_status": "pending",
        "created_on": date.today().isoformat(),
    }

def add_reviewed_candidates(existing, candidates, source_ref, set_version):
    existing_ids = {str(item.get("id")) for item in existing}
    existing_queries = {str(item.get("query", "")).strip() for item in existing}
    accepted = list(existing)
    rejected = []
    for candidate in candidates:
        record = make_candidate_record(candidate, source_ref, set_version, generated_by="llm-candidate")
        valid = validate_eval_case({**candidate, "expected_pages": record["expected_pages"]})
        duplicate = record["id"] in existing_ids or record["query"] in existing_queries
        if valid["ok"] and not valid["has_answer_leak"] and not duplicate:
            accepted.append({**record, "review_status": "pending"})
            existing_ids.add(record["id"]); existing_queries.add(record["query"])
        else:
            rejected.append({"record": record, "reason": {"validation": valid, "duplicate": duplicate}})
    return accepted, rejected

# 生产流程还要由人工把 review_status 改为 approved，并在版本控制中保存变更原因。
sample_existing = [{"id": "old-1", "query": "什么是信息增益？"}]
sample_candidates = [
    {"id": "new-1", "query": "信息增益在决策树中说明什么？", "expected_pages": [12]},
    {"id": "leak-1", "query": "参考答案是什么？", "expected_pages": [12]},
]
sample_added, sample_rejected = add_reviewed_candidates(
    sample_existing, sample_candidates, "page-12", "v-demo"
)
print(
    f"候选审核结果：接受 {len(sample_added)} 条，退回 {len(sample_rejected)} 条；"
    f"新记录状态为{'、'.join(item.get('review_status', '') for item in sample_added[1:])}。"
)

候选审核结果：接受 2 条，退回 1 条；新记录状态为pending。
